# LungCT DirectCP 4-Stage Mid-Ratio Pareto Extension

This notebook combines the prior all-10 LungCT Pareto rows, new DirectCP 4-stage mid-ratio rows, and validated pTVReg reference fields. It displays the screening table and saves the requested Pareto figures.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.reports.lungct_midratio_pareto_extension import (
    DEFAULT_OUTPUT_ROOT,
    METHOD_FDBE,
    METHOD_IDIR,
    METHOD_PTVREG,
    METHOD_PTV_DIRECTCP,
    build_review_tables,
    save_and_plot_review,
)

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
OUTPUT_ROOT = DEFAULT_OUTPUT_ROOT
print(f'Repo root: {REPO_ROOT}')
print(f'Output root: {OUTPUT_ROOT}')

## Editable Exclusion Controls

Edit these dictionaries to exclude points by ratio-to-anchor weight or exact regularization weight. pTVReg is always retained as the reference point.

In [ ]:
EXCLUDE_BY_RATIO = {
    METHOD_IDIR: [],
    METHOD_PTV_DIRECTCP: [0.01, 10.0, 100.0],
    METHOD_FDBE: [0.0, 0.001, 0.01],
}

EXCLUDE_BY_WEIGHT = {
    METHOD_IDIR: [],
    METHOD_PTV_DIRECTCP: [],
    METHOD_FDBE: [],
}

print('EXCLUDE_BY_RATIO =', EXCLUDE_BY_RATIO)
print('EXCLUDE_BY_WEIGHT =', EXCLUDE_BY_WEIGHT)

## Load Rows, Validate pTVReg, And Save Tables

In [ ]:
tables = build_review_tables()
existing_df = tables['existing_df']
extension_df = tables['extension_df']
ptvreg_df = tables['ptvreg_df']
case_df = tables['case_df']
mean_df = tables['mean_df']
table_df = tables['table_df']
completion_errors = tables['completion_errors']

print(f'Existing trained-method case rows: {len(existing_df)}')
print(f'New extension case rows: {len(extension_df)}')
print(f'pTVReg case rows: {len(ptvreg_df)}')
print(f'Combined case rows: {len(case_df)}')
print(f'Combined mean rows: {len(mean_df)}')
if completion_errors:
    raise RuntimeError('Extension completion errors: ' + '; '.join(completion_errors))

display(Markdown('### Extension rows'))
display(extension_df[['method_label', 'case_id', 'reg_value', 'ratio_to_best_tre_weight' if 'ratio_to_best_tre_weight' in extension_df.columns else 'reg_sort', 'original_snap_tre_mean_mm', 'fg_logj_std']])

display(Markdown('### pTVReg field validation'))
display(ptvreg_df[['case_id', 'source_saved_tre_phys_mean_mm', 'source_recomputed_snap_tre_mean_mm', 'source_tre_validation_abs_err', 'source_saved_trestd_case_mm', 'source_recomputed_snap_tre_std_mm', 'source_trestd_validation_abs_err', 'source_disp_h5py_shape']])

display(Markdown('### Screening table'))
display(table_df)

## Pareto Figures

In [ ]:
result = save_and_plot_review(
    case_df,
    mean_df,
    table_df,
    output_root=OUTPUT_ROOT,
    exclude_by_ratio=EXCLUDE_BY_RATIO,
    exclude_by_weight=EXCLUDE_BY_WEIGHT,
)
print('Saved artifacts:')
for key, path in result['artifacts'].items():
    print(f'  {key}: {path}')
print('Output validation:', result['validation'])
if not result['validation']['ok']:
    raise RuntimeError(result['validation'])

## Saved Figure Paths

In [ ]:
for subset, paths in result['plots'].items():
    print(f'[{subset}]')
    for path in paths:
        print(path)